# Bab 15 — Deep learning on unseen data: introducing federated learning

**Mahasiswa:** Tarannia Raisha Putri · 101032300231 · TK4702 · Deep Learning

**Referensi utama:** Andrew W. Trask, *Grokking Deep Learning*, Manning (2019), Bab 15, halaman cetak 281–291 (PDF 302–312, indeks PDF mulai 1). Subbab spesifik dicantumkan pada setiap tahap.

**Tujuan pembelajaran:** Melatih classifier spam, mensimulasikan federated averaging, mereproduksi demonstrasi kebocoran, dan memeriksa agregasi terenkripsi.

## Ringkasan

Federated learning mengirim model ke pemilik data dan menggabungkan hasil latihan lokal. Bab ini memakai klasifikasi spam Enron, menunjukkan kebocoran dari update satu contoh, kemudian menjumlahkan model terenkripsi dengan Paillier. Data tidak dikirim bukan berarti informasi tidak dapat bocor.

## Konsep, intuisi, dan notasi

Untuk email dengan indeks kata $S$, logit $z=\sum_{w\in S}E_w$, $E\in\mathbb R^{V\times1}$, probabilitas $p=\sigma(z)$. Target $(B,1)$. Buku memakai squared error: $L=\sum_i(p_i-y_i)^2$, sehingga gradien logit $2(p-y)p(1-p)$, berbeda dari BCE yang langsung $p-y$.

Federated averaging buku: $\theta_{t+1}=\frac13(\theta_{Bob}+\theta_{Alice}+\theta_{Sue})$. Jika jumlah data berbeda, tambahan weighted averaging memakai $\sum_k(n_k/\sum_jn_j)\theta_k$; dua definisi tidak identik.

Paillier mendukung penjumlahan plaintext melalui operasi ciphertext: $\mathrm{Dec}(\mathrm{Enc}(a)+\mathrm{Enc}(b))=a+b$. Ini enkripsi aditif, bukan kemampuan menjalankan sembarang neural network pada ciphertext. Private key harus terpisah dari individual update pada skenario privasi yang diinginkan. Notebook mensimulasikan semua pihak dalam satu proses untuk memeriksa aljabar, bukan membangun protokol produksi.

## Persiapan dan cara membaca

Jalankan **Restart Kernel → Run All**. Semua penjelasan merupakan parafrasa orisinal; angka hasil berada pada output sel yang benar-benar dijalankan. Label **reproduksi** berarti algoritme/contoh buku ditulis ulang; **koreksi/penyesuaian** menunjukkan perubahan; **tambahan** menunjukkan eksperimen atau interpretasi praktikum. `GDL_MODE=quick` mengurangi latihan berat; `full` memilih anggaran buku, dengan perubahan yang tetap dicatat. Keberhasilan eksekusi tidak membuktikan kesamaan akurasi dengan buku.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Jalankan dari root repository atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODE = os.environ.get('GDL_MODE', 'quick')
assert MODE in {'quick', 'full'}
FULL = MODE == 'full'
np.random.seed(1)
np.set_printoptions(precision=6, suppress=True)
METRICS = {}
def record(**values):
    METRICS.update(values)
def finite(*arrays):
    assert all(np.isfinite(np.asarray(a)).all() for a in arrays), 'NaN/inf terdeteksi'
def softmax(z):
    z = np.asarray(z)
    e = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
def sigmoid(z):
    return np.exp(-np.logaddexp(0, -np.asarray(z)))
print('Mode:', MODE, '| NumPy:', np.__version__)

Mode: quick | NumPy: 2.5.4


<a id="b15-data"></a>

## Reproduksi: privasi dan learning to detect spam

**Sumber:** cetak 282–285; PDF 303–306.

Data `spam.txt` dan `ham.txt` berasal dari snapshot pendamping buku yang merujuk corpus Enron. Tidak ada inbox pribadi yang dibaca. Listing menghapus email dengan ≥500 kata meski narasi menyebut trimming; cabang default di bawah mengikuti filter listing dan melaporkan jumlah yang dikeluarkan. Set kata diurutkan untuk determinisme. Seribu terakhir tiap kelas dipisahkan sebelum memilih subset quick.

Quick mengambil 300 email training tiap kelas dan 100 test tiap kelas; vocabulary dibatasi 1.000 kata training terumum ditambah kata demonstrasi. Full memakai seluruh training yang memenuhi filter, test 2.000, vocabulary penuh training. `<unk>` juga padding dan dibekukan nol seperti buku; kata OOV tidak memberi kontribusi. Ini pembatasan nyata yang tercatat, bukan dataset pengganti.

**Koreksi evaluasi:** konten email test yang juga ada di training dibuang. Jumlah aktual dan duplikat yang dibuang dicetak; test tidak di-oversample setelah deduplikasi.

In [2]:
from src.datasets import fetch
from src.autograd import Tensor,Embedding,SGD,MSELoss
from collections import Counter
import copy

def read_emails(name):
    lines=fetch(name).read_text(encoding='utf-8',errors='ignore').splitlines()
    sets=[sorted(set(line.split())) for line in lines]
    kept=[s for s in sets if len(s)<500]
    print(name,'awal',len(sets),'dipakai filter buku',len(kept),'dikeluarkan',len(sets)-len(kept))
    return kept
spam=read_emails('spam.txt'); ham=read_emails('ham.txt')
train_spam,train_ham=spam[:-1000],ham[:-1000]
test_spam,test_ham=spam[-1000:],ham[-1000:]
if not FULL:
    train_spam,train_ham=train_spam[:300],train_ham[:300]
    test_spam,test_ham=test_spam[:100],test_ham[:100]
training_content=set(map(tuple,train_spam+train_ham))
before_test=len(test_spam)+len(test_ham)
test_spam=[s for s in test_spam if tuple(s) not in training_content]
test_ham=[s for s in test_ham if tuple(s) not in training_content]
duplicates_removed=before_test-len(test_spam)-len(test_ham)
print('Koreksi tambahan: duplikat konten test terhadap train dibuang:',duplicates_removed)
counts=Counter(w for s in train_spam+train_ham for w in s)
selected=sorted(counts) if FULL else [w for w,_ in sorted(counts.items(),key=lambda item:(-item[1],item[0]))[:1000]]
vocab=['<unk>']+sorted(set(selected)|{'my','computer','password','is','pizza'})
w2i={w:i for i,w in enumerate(vocab)}
def encode(sentences):
    data=np.zeros((len(sentences),500),dtype=int)
    for i,sentence in enumerate(sentences):
        ids=[w2i.get(w,0) for w in sentence[:500]]; data[i,:len(ids)]=ids
    return data
def balanced(a,b):
    # Pengulangan kelas yang lebih kecil mengikuti interleaving listing buku.
    rows=[]; targets=[]
    for i in range(max(len(a),len(b))):
        rows.extend([a[i%len(a)],b[i%len(b)]]); targets.extend([[1.],[0.]])
    return encode(rows),np.array(targets)
train_data,train_target=balanced(train_spam,train_ham)
test_data=encode(test_spam+test_ham)
test_target=np.array([[1.]]*len(test_spam)+[[0.]]*len(test_ham))
print('Shapes train/test:',train_data.shape,test_data.shape,'Vocab:',len(vocab))
assert not set(map(tuple,train_spam)) & set(map(tuple,test_spam)), 'Duplikat lintas split spam; perlu audit'
assert not set(map(tuple,train_spam+train_ham)) & set(map(tuple,test_spam+test_ham))

spam.txt awal 9000 dipakai filter buku 8766 dikeluarkan 234


ham.txt awal 22032 dipakai filter buku 21954 dikeluarkan 78
Koreksi tambahan: duplikat konten test terhadap train dibuang: 1
Shapes train/test: (600, 500) (199, 500) Vocab: 1003


<a id="b15-central"></a>

## Reproduksi: centralized training dan test

**Sumber:** cetak 285–285; PDF 306–306.

`batch_size` menggantikan variabel `bs` yang tidak didefinisikan pada listing. Optimizer dibentuk terhadap model yang diterima fungsi, sehingga tidak salah memperbarui model global saat melatih salinan lokal. MSE mean dikalikan jumlah contoh untuk mempertahankan squared-error sum buku. Padding dinolkan sebelum forward dan setelah update; gradien padding juga dinolkan.

In [3]:
def new_model():
    model=Embedding(len(vocab),1); model.weight.data*=0; return model

def train(model,data,target,batch_size=None,epochs=1):
    batch_size=batch_size or (500 if FULL else 64)
    optimizer=SGD(model.get_parameters(),.01); losses=[]
    for epoch in range(epochs):
        total=0.
        for start in range(0,len(data),batch_size):
            xb=data[start:start+batch_size]; yb=target[start:start+batch_size]
            model.weight.data[0]=0
            pred=model(xb).sum(1).sigmoid()
            loss=MSELoss()(pred,Tensor(yb))*len(xb)
            loss.backward(); model.weight.grad[0]=0; optimizer.step(); model.weight.data[0]=0
            total+=loss.data.item()
        losses.append(total/len(data))
    return model,losses

def evaluate(model,data,target):
    model.weight.data[0]=0
    probabilities=sigmoid(model.weight.data[data].sum(axis=1))
    finite(probabilities,model.weight.data)
    return float(np.mean((probabilities>.5)==target))
central=new_model()
for epoch in range(3):
    central,losses=train(central,train_data,train_target)
    print('Central epoch',epoch+1,'MSE',losses[-1],'test accuracy',evaluate(central,test_data,test_target))

Central epoch 1 MSE 0.15567762460058737 test accuracy 0.9447236180904522
Central epoch 2 MSE 0.08193071421763413 test accuracy 0.9447236180904522
Central epoch 3 MSE 0.06321406548620927 test accuracy 0.9447236180904522


<a id="b15-federated"></a>

## Reproduksi: Bob, Alice, Sue dan averaging

**Sumber:** cetak 286–286; PDF 307–307.

Setiap peserta menerima salinan bobot awal ronde yang sama. Tidak boleh memperbarui model global setelah Bob lalu mengirim hasil Bob ke Alice: itu sequential training, bukan agregasi ronde. Rata-rata setara buku tetap dipakai; weighted averaging ditampilkan sebagai perbandingan tambahan jika ukuran shard berbeda.

In [4]:
if FULL:
    partitions=[np.arange(0,1000),np.arange(1000,2000),np.arange(2000,len(train_data))]
else:
    partitions=np.array_split(np.arange(len(train_data)),3)
assert len(np.unique(np.concatenate(partitions)))==len(train_data)
clients=[(train_data[idx],train_target[idx]) for idx in partitions]
federated=new_model()
for round_id in range(3):
    local=[train(copy.deepcopy(federated),x,y)[0] for x,y in clients]
    weights=np.stack([m.weight.data for m in local])
    equal=weights.mean(0)
    weighted=np.average(weights,axis=0,weights=[len(x) for x,y in clients])
    federated.weight.data[:]=equal
    print('Round',round_id+1,'client sizes',[len(x) for x,y in clients], 'test accuracy',evaluate(federated,test_data,test_target),'equal-weighted distance',np.linalg.norm(equal-weighted))

Round 1 client sizes [200, 200, 200] test accuracy 0.9447236180904522 equal-weighted distance 4.79824195159329e-17
Round 2 client sizes [200, 200, 200] test accuracy 0.9447236180904522 equal-weighted distance 8.377717605484208e-17
Round 3 client sizes [200, 200, 200] test accuracy 0.9447236180904522 equal-weighted distance 1.1233237820841046e-16


<a id="b15-leak"></a>

## Reproduksi: kebocoran update satu email

**Sumber:** cetak 287–287; PDF 308–308.

Contoh kalimat buatan Bob memang dari buku. Bob mengirim update untuk satu email; baris embedding yang berubah mengungkap **himpunan kata**, bukan urutan lengkap. Itu cukup menunjukkan federated learning sendiri tidak menjamin kerahasiaan. Demonstrasi hanya memakai kalimat contoh, bukan data pribadi pengguna.

In [5]:
message=['my','computer','password','is','pizza']
original=new_model()
bob_input=np.array([[w2i[w] for w in message]])
bob,_=train(copy.deepcopy(original),bob_input,np.array([[0.]]),batch_size=1)
changed=np.flatnonzero(np.any(bob.weight.data!=original.weight.data,axis=1))
revealed=[vocab[i] for i in changed]
print('Kata yang terungkap:',revealed)
assert set(revealed)==set(message)

Kata yang terungkap: ['computer', 'is', 'my', 'password', 'pizza']


<a id="b15-crypto"></a>

## Reproduksi: homomorphic encryption dan secure aggregation

**Sumber:** cetak 288–291; PDF 309–312.

Contoh 5+3 memakai library `phe`. Agregasi vektor memakai key 128 bit seperti demonstrasi buku agar terjangkau; **ukuran ini hanya untuk pengajaran, tidak aman untuk sistem nyata**. Private key/ciphertext tidak dicetak atau disimpan. Sifat acak kriptografi tidak dipaksa seed tetap; bobot plaintext tetap deterministik.

Koreksi utama halaman 290: dekripsi dilakukan pada `aggregated_model`, bukan hanya `sue_encrypted_model`. Ketiga peserta menjumlahkan ciphertext sebelum hasil agregat didekripsi. Dalam simulasi satu proses ini pembatasan akses pihak hanya konseptual; secure aggregation tidak sama dengan differential privacy dan tidak melindungi dari semua bentuk kolusi.

In [6]:
from phe import paillier
public,private=paillier.generate_paillier_keypair(n_length=1024)
answer=private.decrypt(public.encrypt(5)+public.encrypt(3))
print('Penjumlahan terenkripsi 5+3:',answer); assert answer==8
public,private=paillier.generate_paillier_keypair(n_length=128)
def train_and_encrypt(model,x,y,pubkey):
    trained,_=train(copy.deepcopy(model),x,y)
    encrypted=np.array([pubkey.encrypt(float(v)) for v in trained.weight.data[:,0]],dtype=object)
    return encrypted,trained.weight.data.copy()
secure=new_model(); aggregate_errors=[]
for round_id in range(3):
    clients_result=[train_and_encrypt(secure,x,y,public) for x,y in clients]
    aggregated=clients_result[0][0]+clients_result[1][0]+clients_result[2][0]
    decoded=np.array([private.decrypt(value) for value in aggregated])[:,None]/3
    expected=np.mean([item[1] for item in clients_result],axis=0)
    error=float(np.max(np.abs(decoded-expected))); aggregate_errors.append(error)
    np.testing.assert_allclose(decoded,expected,atol=1e-10)
    secure.weight.data[:]=decoded
    print('Secure round',round_id+1,'error agregasi',error,'test accuracy',evaluate(secure,test_data,test_target))
record(train_count=len(train_data),test_count=len(test_data),test_duplicates_removed=duplicates_removed,vocab_size=len(vocab),centralized_test_accuracy=evaluate(central,test_data,test_target),federated_test_accuracy=evaluate(federated,test_data,test_target),secure_test_accuracy=evaluate(secure,test_data,test_target),aggregate_max_error=max(aggregate_errors),revealed_word_count=len(revealed))

Penjumlahan terenkripsi 5+3: 8


Secure round 1 error agregasi 1.3877787807814457e-17 test accuracy 0.9447236180904522


Secure round 2 error agregasi 2.7755575615628914e-17 test accuracy 0.9447236180904522


Secure round 3 error agregasi 2.7755575615628914e-17 test accuracy 0.9447236180904522


## Analisis output aktual — run tersimpan

Mode **quick**, eksekusi 2026-10-16T01:51:21.636955+07:00. Ringkasan ini dibuat dari output validator, bukan angka contoh dalam buku.

Dengan 600 baris train dan 199 test, akurasi centralized 94.47%, federated 94.47%, dan agregasi terenkripsi 94.47%. Sebanyak 1 duplikat test terhadap train dibuang. Galat maksimum decrypt-average 2.776e-17; kesamaan aritmetika ini tidak membuktikan protokol aman untuk produksi. Demonstrasi satu email mengungkap 5 kata.

## Kesimpulan dan pembahasan pertanyaan

Seluruh tahapan inti dilaksanakan: centralized, federated, kebocoran, aritmetika terenkripsi, dan agregasi ronde. Kesetaraan ciphertext/plaintext diperiksa numerik. Skor quick memakai subset dan vocabulary lebih kecil; bukan reproduksi angka 99% pada buku.

**Mengapa tidak cukup menyembunyikan data mentah?** Update model masih membawa sinyal tentang data yang menghasilkannya.

**Apa bug dekripsi Sue saja?** Model menjadi sepertiga hasil satu peserta, bukan rata-rata tiga peserta.

**Apakah enkripsi menghilangkan kebocoran dari model akhir?** Tidak; ia melindungi operasi/komunikasi tertentu. Privasi hasil akhir memerlukan asumsi dan mekanisme tambahan.

**Apakah Paillier menjamin server tidak melihat update individual di notebook?** Tidak, notebook adalah simulasi satu proses; protokol nyata harus menegakkan batas akses dan asumsi kepercayaan.

In [7]:
finite(*[v for v in METRICS.values() if isinstance(v, (int, float))])
print('VALIDATION_METRICS=' + json.dumps(METRICS, sort_keys=True))

VALIDATION_METRICS={"aggregate_max_error": 2.7755575615628914e-17, "centralized_test_accuracy": 0.9447236180904522, "federated_test_accuracy": 0.9447236180904522, "revealed_word_count": 5, "secure_test_accuracy": 0.9447236180904522, "test_count": 199, "test_duplicates_removed": 1, "train_count": 600, "vocab_size": 1003}
